# Feature Engineering

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os

sys.path.append(os.path.abspath('../src'))
from dataloader import Dataloader

dl = Dataloader()
DATA_DIR = Path("../data")

def load_local_data():
    data = {}
    files = ["vendas", "filiais", "metas", "devolucoes"]
    for f in files:
        path = DATA_DIR / f"{f}.parquet"
        if path.exists():
            data[f] = pd.read_parquet(path)
        else:
            print(f"Warning: {f}.parquet not found in {DATA_DIR}")
    return data

datasets = load_local_data()
vendas = datasets.get('vendas')
filiais = datasets.get('filiais')
metas = datasets.get('metas')
devolucoes = datasets.get('devolucoes')

In [16]:
if metas is not None:
    metas['meta_med'] = metas['meta_med'].clip(lower=0)
    metas['meta_n_med'] = metas['meta_n_med'].clip(lower=0)

In [17]:
if vendas is not None:
    vendas['data_emissao'] = pd.to_datetime(vendas['data_emissao'])
    
    daily_sales = vendas.groupby(['data_emissao', 'codigo_filial', 'categoria_gerencial'])['faturamento'].sum().reset_index()
    
    daily_sales['day_of_week'] = daily_sales['data_emissao'].dt.dayofweek
    daily_sales['month'] = daily_sales['data_emissao'].dt.month
    daily_sales['day_of_month'] = daily_sales['data_emissao'].dt.day
    daily_sales['is_weekend'] = daily_sales['day_of_week'].isin([5, 6]).astype(int)
    
    daily_sales['quarter'] = daily_sales['data_emissao'].dt.quarter
    daily_sales['is_month_end'] = daily_sales['data_emissao'].dt.is_month_end.astype(int)
    
    print("Shape:", daily_sales.shape)

Shape: (159886, 10)


In [18]:
if filiais is not None and 'daily_sales' in locals():
    binary_cols = ['delivery', 'panvel_clinic', 'estacionamento', 'atendimento_24_horas']
    for col in binary_cols:
        if col in filiais.columns:
            filiais[col] = filiais[col].map({'S': 1, 'N': 0}).fillna(0)
            
    df_final = daily_sales.merge(filiais, on='codigo_filial', how='left')
    
    df_final['categoria_id'] = df_final['categoria_gerencial'].astype('category').cat.codes
    

In [19]:
def create_lags(df):
    df = df.sort_values(['codigo_filial', 'categoria_gerencial', 'data_emissao'])
    group = df.groupby(['codigo_filial', 'categoria_gerencial'])['faturamento']
    
    df['sales_lag_1'] = group.shift(1)
    df['sales_lag_7'] = group.shift(7)
    
    df['sales_avg_7d'] = group.transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
    df['sales_avg_30d'] = group.transform(lambda x: x.shift(1).rolling(window=30, min_periods=1).mean())
    
    return df

df_final = create_lags(df_final)

In [20]:
df_final = df_final.fillna(0)

output_path = "../data/processed_features.csv"
df_final.to_csv(output_path, index=False)
display(df_final.head())

,data_emissao,codigo_filial,categoria_gerencial,faturamento,day_of_week,month,day_of_month,is_weekend,quarter,is_month_end,...,delivery,metragem_area_venda,panvel_clinic,estacionamento,atendimento_24_horas,categoria_id,sales_lag_1,sales_lag_7,sales_avg_7d,sales_avg_30d
166,2024-01-02 00:00:00+00:00,1500,MED,31029.20,1,1,2,0,1,0,...,0.0,309.0388,0.0,0.0,0.0,0,0.00,0.0,0.000000,0.000000
368,2024-01-03 00:00:00+00:00,1500,MED,34690.13,2,1,3,0,1,0,...,0.0,309.0388,0.0,0.0,0.0,0,31029.20,0.0,31029.200000,31029.200000
570,2024-01-04 00:00:00+00:00,1500,MED,37898.29,3,1,4,0,1,0,...,0.0,309.0388,0.0,0.0,0.0,0,34690.13,0.0,32859.665000,32859.665000
772,2024-01-05 00:00:00+00:00,1500,MED,40694.85,4,1,5,0,1,0,...,0.0,309.0388,0.0,0.0,0.0,0,37898.29,0.0,34539.206667,34539.206667
974,2024-01-06 00:00:00+00:00,1500,MED,20451.83,5,1,6,1,1,0,...,0.0,309.0388,0.0,0.0,0.0,0,40694.85,0.0,36078.117500,36078.117500
